# Satellite Image → Economic Activity Predictor
## Interactive Demo Notebook

This notebook demonstrates the complete workflow for predicting economic activity from satellite imagery using deep learning.

### What We'll Cover:
1. 📦 **Data Generation** - Create synthetic satellite imagery
2. 📊 **Data Exploration** - Visualize data distributions
3. 🧠 **Model Training** - Train a deep learning model
4. 📈 **Evaluation** - Assess model performance
5. 🔮 **Prediction** - Make predictions on new images

## 1. Setup and Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our modules
from download_data import SatelliteDataDownloader
from dataset import create_dataloaders
from model import get_model
from train import Trainer
from evaluate import ModelEvaluator
from predict import EconomicPredictor
from visualize import *
from utils import set_seed, setup_logging

# Setup
setup_logging()
set_seed(42)

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Generate Sample Data

We'll generate synthetic satellite images with corresponding economic indicators.

In [ ]:
# Generate sample data
downloader = SatelliteDataDownloader(output_dir='../data/raw')
df = downloader.download_sample_data(num_samples=100)

print(f"\nGenerated {len(df)} samples")
print("\nFirst few rows:")
df.head()

## 3. Explore the Data

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Visualize sample images
plot_sample_images(
    image_dir='../data/raw/images',
    labels_file='../data/raw/labels.csv',
    num_samples=9,
    save_path='../outputs/sample_images.png'
)

In [ ]:
# Distribution of economic indicators
plot_data_distribution(
    labels_file='../data/raw/labels.csv',
    save_path='../outputs/data_distribution.png'
)

In [ ]:
# Correlation matrix
plot_correlation_matrix(
    labels_file='../data/raw/labels.csv',
    save_path='../outputs/correlation_matrix.png'
)

## 4. Prepare Data Loaders

In [ ]:
# Configuration
config = {
    'backbone': 'resnet50',
    'pretrained': True,
    'num_indicators': 5,
    'dropout_rate': 0.3,
    'hidden_dim': 512,
    'batch_size': 16,
    'num_epochs': 10,  # Reduced for demo
    'learning_rate': 0.0001,
    'weight_decay': 0.00001,
    'optimizer': 'adamw',
    'scheduler': 'plateau',
    'early_stopping_patience': 5,
    'use_amp': True,
    'data_dir': '../data/raw/images',
    'labels_file': '../data/raw/labels.csv',
    'model_dir': '../models',
    'output_dir': '../outputs',
    'image_size': 224,
    'num_workers': 2
}

# Create dataloaders
dataloaders = create_dataloaders(
    data_dir=config['data_dir'],
    labels_file=config['labels_file'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    image_size=config['image_size']
)

print("Data splits:")
print(f"  Train: {len(dataloaders['train'].dataset)} samples")
print(f"  Val:   {len(dataloaders['val'].dataset)} samples")
print(f"  Test:  {len(dataloaders['test'].dataset)} samples")

## 5. Build and Inspect Model

In [ ]:
# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = get_model(config)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture: {config['backbone']}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1024**2:.2f} MB (FP32)")

In [ ]:
# Test forward pass
sample_batch = next(iter(dataloaders['train']))
sample_images, sample_labels = sample_batch

print(f"Sample batch shape: {sample_images.shape}")

model.eval()
with torch.no_grad():
    sample_predictions = model(sample_images)

print("\nModel outputs:")
for task, pred in sample_predictions.items():
    print(f"  {task}: {pred.shape}")

## 6. Train the Model

⚠️ **Note**: This may take several minutes depending on your hardware.

In [ ]:
# Create trainer
trainer = Trainer(
    config=config,
    model=model,
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    device=device
)

print("Starting training...")
print(f"Epochs: {config['num_epochs']}")
print(f"Batch size: {config['batch_size']}")
print("-" * 60)

In [ ]:
# Train the model
trainer.train(num_epochs=config['num_epochs'])

In [ ]:
# Plot training history
from IPython.display import Image, display
display(Image('../outputs/training_history.png'))

## 7. Evaluate on Test Set

In [ ]:
# Load best model
best_model_path = Path('../models/best_model.pth')
if best_model_path.exists():
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded best model")

# Evaluate
evaluator = ModelEvaluator(model, device=device)
metrics = evaluator.evaluate(dataloaders['test'])

# Display results
print("\n" + "="*70)
print("TEST SET EVALUATION RESULTS")
print("="*70)

for task_name, task_metrics in metrics.items():
    print(f"\n{task_name.upper().replace('_', ' ')}:")
    print("-"*70)
    for metric_name, value in task_metrics.items():
        print(f"  {metric_name:30s}: {value:10.4f}")

In [ ]:
# Convert metrics to DataFrame for better visualization
metrics_df = pd.DataFrame(metrics).T
metrics_df.style.background_gradient(cmap='RdYlGn_r', subset=['MAE', 'RMSE', 'MAPE']) \
    .background_gradient(cmap='RdYlGn', subset=['R2', 'Pearson_Correlation'])

## 8. Visualize Predictions

In [ ]:
# Get predictions for test set
all_predictions = {
    'economic_activity': [],
    'nightlight_intensity': [],
    'building_density': [],
    'road_density': [],
    'vegetation_index': []
}

all_targets = {
    'economic_activity': [],
    'nightlight_intensity': [],
    'building_density': [],
    'road_density': [],
    'vegetation_index': []
}

model.eval()
with torch.no_grad():
    for images, labels in dataloaders['test']:
        images = images.to(device)
        predictions = model(images)
        
        for task in all_predictions.keys():
            all_predictions[task].extend(predictions[task].cpu().numpy())
            all_targets[task].extend(labels[task].cpu().numpy())

# Convert to numpy
all_predictions = {k: np.array(v) for k, v in all_predictions.items()}
all_targets = {k: np.array(v) for k, v in all_targets.items()}

In [ ]:
# Create prediction plots
plot_model_predictions_comparison(
    predictions=all_predictions,
    targets=all_targets,
    save_path='../outputs/prediction_comparison.png'
)

## 9. Make Predictions on New Images

In [ ]:
# Create predictor
predictor = EconomicPredictor(
    model_path='../models/best_model.pth',
    device=str(device)
)

# Get a test image
test_df = pd.read_csv('../data/raw/test_labels.csv')
sample_image = test_df.iloc[0]['image_filename']
sample_image_path = f"../data/raw/images/{sample_image}"

print(f"Making prediction on: {sample_image}")

# Predict
predictions = predictor.predict_image(sample_image_path)

# Display
print("\nPredictions:")
print("-" * 60)
for task, value in predictions.items():
    actual_value = test_df.iloc[0][task]
    error = abs(value - actual_value)
    print(f"{task.replace('_', ' ').title():30s}")
    print(f"  Predicted: {value:8.3f}")
    print(f"  Actual:    {actual_value:8.3f}")
    print(f"  Error:     {error:8.3f}")
    print()

In [ ]:
# Visualize prediction
predictor.predict_and_visualize(
    image_path=sample_image_path,
    save_path='../outputs/sample_prediction.png'
)

## 10. Analysis: Feature Importance

Let's analyze which parts of the satellite images the model focuses on.

In [ ]:
# Analyze multiple predictions
comparison_df = pd.DataFrame()

for i in range(min(10, len(test_df))):
    row = test_df.iloc[i]
    img_path = f"../data/raw/images/{row['image_filename']}"
    
    pred = predictor.predict_image(img_path)
    
    for task in pred.keys():
        comparison_df.loc[i, f'{task}_pred'] = pred[task]
        comparison_df.loc[i, f'{task}_actual'] = row[task]
        comparison_df.loc[i, f'{task}_error'] = abs(pred[task] - row[task])

print("\nPrediction vs Actual Comparison (First 10 samples):")
comparison_df.head(10)

In [ ]:
# Plot error distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

tasks = ['economic_activity', 'nightlight_intensity', 'building_density', 
         'road_density', 'vegetation_index']

for idx, task in enumerate(tasks):
    error_col = f'{task}_error'
    if error_col in comparison_df.columns:
        axes[idx].hist(comparison_df[error_col], bins=20, alpha=0.7, 
                      color='steelblue', edgecolor='black')
        axes[idx].set_xlabel('Absolute Error', fontsize=12)
        axes[idx].set_ylabel('Frequency', fontsize=12)
        axes[idx].set_title(f'{task.replace("_", " ").title()}\n'
                          f'Mean Error: {comparison_df[error_col].mean():.3f}',
                          fontsize=13)
        axes[idx].grid(True, alpha=0.3)

fig.delaxes(axes[-1])
plt.tight_layout()
plt.savefig('../outputs/error_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Summary and Next Steps

### What We've Accomplished:
✅ Generated synthetic satellite imagery with economic indicators  
✅ Trained a deep learning model to predict economic activity  
✅ Evaluated model performance on test set  
✅ Made predictions on new images  

### Potential Improvements:
1. **Use Real Data**: Replace synthetic data with actual satellite imagery (Landsat, Sentinel-2)
2. **More Training**: Increase epochs and dataset size
3. **Hyperparameter Tuning**: Optimize learning rate, batch size, model architecture
4. **Ensemble Methods**: Combine multiple models for better predictions
5. **Temporal Analysis**: Include time-series data for trend prediction
6. **Transfer Learning**: Fine-tune on specific regions or economic sectors

### Real-World Applications:
- 💰 Investment firms tracking economic development
- 🏛️ Government agencies monitoring regional growth
- 🌍 Development organizations assessing aid impact
- 📊 Economic research and forecasting

## 12. Export Model for Production

In [ ]:
# Export to ONNX for deployment
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

onnx_path = '../models/economic_predictor.onnx'

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['satellite_image'],
        output_names=list(all_predictions.keys()),
        dynamic_axes={
            'satellite_image': {0: 'batch_size'},
        }
    )
    print(f"✅ Model exported to ONNX: {onnx_path}")
except Exception as e:
    print(f"⚠️ ONNX export failed: {e}")
    print("This is optional - the PyTorch model can still be used.")

In [ ]:
# Save configuration
import json

config_export = {
    'model_info': {
        'architecture': config['backbone'],
        'input_size': [3, 224, 224],
        'output_tasks': list(all_predictions.keys()),
        'total_parameters': total_params
    },
    'performance': {
        task: {metric: float(value) for metric, value in metrics[task].items()}
        for task in all_predictions.keys()
    },
    'training_config': config
}

with open('../models/model_config.json', 'w') as f:
    json.dump(config_export, f, indent=2)

print("✅ Configuration saved to models/model_config.json")